# 📝 Week 4 Homework: Data Wrangling - From Business Question to Analysis

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-7025/blob/main/assignments/homework/week-04-homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## 📂 Instructions

This homework is based on the Lab we worked through in Thursday's class.  So, if you completed that Lab, you can use that notebook for the homework.

Complete the tasks below in this Jupyter notebook. Most tasks require you to write Python code and use the output to answer **a separate online quiz**.

At the end, you’ll also upload this completed `.ipynb` notebook

---

In this homework, we’ll use **three datasets** from the Complete Journey retail grocery data:

1. **transactions** – product purchases by households (receipt-level detail)  
2. **demographics** – household-level demographic data  
3. **products** – metadata about products purchased  

This homework reinforces this week’s readings:

- **[Reading 10: Manipulating Data](https://bradleyboehmke.github.io/uc-bana-7025/10-manipulating-data.html)**
- **[Reading 11: Summarizing Data](https://bradleyboehmke.github.io/uc-bana-7025/11_aggregating_data.html)**
- **[Reading 12: Joining Data](https://bradleyboehmke.github.io/uc-bana-7025/12-joining-data.html)**

We will:
- Start with simple data exploration
- Progress to manipulating and summarizing data
- End with joining datasets to answer more complex questions
- Practice breaking business questions into **analytical steps**

You are encouraged to work in small groups of **2–4 students** but you must submit your own notebook.


## Setup

In [63]:
!pip install completejourney-py
# If you don't have completejourney_py installed, run: pip install completejourney-py
from completejourney_py import get_data
import pandas as pd

# Load datasets
cj_data = get_data()
transactions = cj_data['transactions']
products = cj_data['products']
demographics = cj_data['demographics']

# Quick preview
transactions.head()

#demographics.head()

#products.head()


,household_id,store_id,basket_id,product_id,quantity,sales_value,retail_disc,coupon_disc,coupon_match_disc,week,transaction_timestamp
0,900,330,31198570044,1095275,1,0.50,0.00,0.0,0.0,1,2017-01-01 11:53:26
1,900,330,31198570047,9878513,1,0.99,0.10,0.0,0.0,1,2017-01-01 12:10:28
2,1228,406,31198655051,1041453,1,1.43,0.15,0.0,0.0,1,2017-01-01 12:26:30
3,906,319,31198705046,1020156,1,1.50,0.29,0.0,0.0,1,2017-01-01 12:30:27
4,906,319,31198705046,1053875,2,2.78,0.80,0.0,0.0,1,2017-01-01 12:30:27


## Part 1 – Basic Exploration


**Q0:** How many transactions are in our dataset, what is the date range, how many households have demographic data, how many products exist, and what are the min/max/mean sales values?  

**Step-by-step instructions:**
1. Use `.shape[0]` on `transactions` to count rows.  
2. Use `.min()` and `.max()` on `transaction_timestamp` to find the date range.  
3. Use `.shape[0]` on `demographics` and `products` to get counts.  
4. Use `.min()`, `.max()`, `.mean()` on `sales_value` for basic stats.


In [64]:
# Starter code with blanks to fill
# total number of transactions
print(transactions.shape)
num_transactions = transactions.shape[0]

(1469307, 11)


In [65]:
# date range of transactions
min_date = transactions['transaction_timestamp'].min()
max_date = transactions['transaction_timestamp'].max()

print(min_date)
print(max_date)

2017-01-01 11:53:26
2018-01-01 04:01:20


In [68]:
# number of unique households and products
num_households = demographics.shape[0]
num_products = products.shape[0]

print(num_households)
print(num_products)

801
92331


In [69]:
# summary statistics for sales_value
min_sales = transactions['sales_value'].min()
max_sales = transactions['sales_value'].max()
mean_sales = transactions['sales_value'].mean()

print(min_sales)
print(max_sales)
print(mean_sales)

0.0
840.0
3.12803218115751



**Q1:** Which day had the highest total sales?  

**Step-by-step instructions:**
1. Create a new column `date` by extracting only the date from `transaction_timestamp` (`.dt.date`).  
2. Group by `date` and sum `sales_value`.  
3. Sort results in descending order.  
4. Select the top row.


In [70]:
# Your code here
transactions['date'] = transactions['transaction_timestamp'].dt.date

transactions.groupby('date') \
  .agg({'sales_value': 'sum'}) \
  .sort_values(by='sales_value', ascending=False) \
  .head(1)



,sales_value
date,
2017-12-23,24994.47



**Q2:** What are the top 5 departments by total sales?  

**Step-by-step instructions:**
1. Join `transactions` to `products` on `product_id` using an inner join.  
2. Group by `department` and sum `sales_value`.  
3. Sort results in descending order.  
4. Display the top 5.


In [71]:
# Your code here

pd.merge(transactions, products, how="inner", on="product_id") \
  .groupby("department") \
  .agg({"sales_value":"sum"}) \
  .sort_values("sales_value", ascending=False) \
  .head(5)



,sales_value
department,
GROCERY,2316393.89
DRUG GM,596827.45
FUEL,329594.45
PRODUCE,322858.82
MEAT,308575.33


## Part 2 – Manipulating Data


**Q3:** What is the average unit price for each department?  

**Step-by-step instructions:**
1. Create a `unit_price` column: `sales_value / quantity`.  
2. Join `transactions` to `products` to bring in `department`.  
3. Group by `department` and calculate the mean of `unit_price`.


In [72]:
# Your code here
transactions["unit_price"] = transactions["sales_value"] / transactions["quantity"]
transactions["unit_price"].describe()

# I'm intentionally leaving in missing values for now (see next question)

pd.merge(transactions, products, how="inner", on="product_id") \
  .groupby("department") \
  .agg({"unit_price":"mean"})


,unit_price
department,
AUTOMOTIVE,7.216111
CHEF SHOPPE,2.522274
CNTRL/STORE SUP,3.150000
COSMETICS,4.138923
COUPON,1.296070
DELI,inf
DRUG GM,inf
ELECT &PLUMBING,1.000000
FLORAL,7.732635



**Q4:** Do we have missing values in `unit_price`?  

**Step-by-step instructions:**
1. Use `.isna().sum()` on `unit_price` to count missing values.  
2. Consider filtering rows where `quantity == 0` to see if that’s the cause.


In [73]:
# Your code here
if transactions["unit_price"].isna().sum() > 0:
  print("%s missing values" % transactions["unit_price"].isna().sum())

print("Before filtering: %s" % transactions.shape[0])

transactions2 = transactions.loc[transactions['quantity'] > 0]
print("After filtering : %s" % transactions2.shape[0])


8820 missing values
Before filtering: 1469307
After filtering : 1460438


## Part 3 – Aggregations


**Q5:** Which income level spends the most on average?

*Hint:* Join transactions to demographics, group by income, calculate mean sales per income level.


In [80]:
# Your code here
pd.merge(transactions, demographics, how="inner", on="household_id") \
  .groupby("income") \
  .agg({"sales_value":"mean"}) \
  .sort_values("sales_value", ascending=False) \
  .head(5)


,sales_value
income,
175-199K,3.754513
250K+,3.724832
200-249K,3.703222
150-174K,3.541206
100-124K,3.481148



**Q6:** Do households with kids spend more (on average) than households without kids?  

*Hint:* Use `kid_count` to group households by creating a new column (e.g., `has_kids`) that identifies whether a household has kids (`kid_count > 0`) or not (`kid_count == 0`). Note that the tricky part of this step is that `kid_count` is not a numeric variable 🤔. Compute the average spend for those with kids and those without.


In [82]:
# Your code here
demographics['has_kids'] = demographics['kids_count'].fillna(0) != '0'
pd.merge(transactions, demographics, how="inner", on="household_id") \
  .groupby("has_kids") \
  .agg({"sales_value":"mean"})

,sales_value
has_kids,
False,3.177327
True,3.150616



**Q7:** What are the top 5 departments by total quantity of items sold?  

*Hint:* Join to products, group by department, sum quantity, and sort.


In [83]:
# Your code here
pd.merge(transactions, products, how="inner", on="product_id") \
  .groupby("department") \
  .agg({"quantity":"sum"}) \
  .sort_values("quantity", ascending=False) \
  .head(5)


,quantity
department,
FUEL,129662940
MISCELLANEOUS,21361882
GROCERY,1242944
DRUG GM,198635
PRODUCE,185444


## Part 4 – Joins for Deeper Insights


**Q8:** Which product is purchased most frequently?  

*Hint:* Group by `product_id`, sum quantity, then join to products for description.


In [87]:
# Your code here
transactions.groupby("product_id")\
  .agg({"quantity":"sum"})\
  .merge(products, on="product_id", how="inner")\
  .sort_values("quantity", ascending=False)\
  .head(1)


,product_id,quantity,manufacturer_id,department,brand,product_category,product_type,package_size
42352,6534178,126868510,69,FUEL,Private,COUPON/MISC ITEMS,GASOLINE-REG UNLEADED,None



**Q9:** Identify all products with “pizza” in `product_type` and find the one with the greatest total sales.  

*Hint:* Filter products where product_type contains "pizza" with `.str.contains("pizza", case=False, na=False)`, join to transactions, sum sales by product.


In [92]:
# Your code here
pizza = products.loc[products["product_type"].str.contains("pizza", case=False, na=False)]
transactions.groupby("product_id")\
  .agg({"sales_value":"sum"})\
  .merge(pizza, on="product_id", how="inner")\
  .sort_values("sales_value", ascending=False)\
  .head(5)



,product_id,sales_value,manufacturer_id,department,brand,product_category,product_type,package_size
97,944139,1344.50,1755,GROCERY,National,FROZEN PIZZA,PIZZA/TRADITIONAL,15.7 OZ
70,906838,1263.78,1722,GROCERY,National,FROZEN PIZZA,PIZZA/PREMIUM,28.30 OZ
357,12648296,1230.36,1722,GROCERY,National,FROZEN PIZZA,PIZZA/TRADITIONAL,21.6 OZ
113,969568,1125.18,1755,GROCERY,National,FROZEN PIZZA,PIZZA/TRADITIONAL,22 OZ
85,925626,1017.60,1039,GROCERY,National,FROZEN PIZZA,PIZZA/ECONOMY,10.2 OZ



**Q10:** Which product category brings in the most revenue for the highest-income households with kids?

*Hint:* Filter demographics for the highest income level & `kid_count > 0`, join to transactions and products, group by category and compute the sum of sales value.


In [105]:
# Your code here
high_income_kids = demographics.loc[(demographics["income"] == '250K+') & (demographics["has_kids"])]
transactions.merge(high_income_kids, on="household_id", how="inner")\
  .merge(products, on="product_id", how="inner")\
  .groupby("product_category")\
  .agg({"sales_value":"sum"})\
  .sort_values("sales_value", ascending=False)\
  .head(5)


,sales_value
product_category,
COUPON/MISC ITEMS,1415.99
BEEF,705.56
SOFT DRINKS,495.30
FLUID MILK PRODUCTS,457.75
DOMESTIC WINE,440.67



**Q11:** Which manufacturer has the highest total sales, and which department do they primarily sell in?  

*Hint:* Join transactions to products, group by manufacturer, sum sales, find top. Then, filter products for that top manufacturer and check which department(s) they are associated with.


In [138]:
# Your code here
man_trans = transactions.merge(products, on="product_id", how="inner")\
  .groupby("manufacturer_id", as_index=False)\
  .agg({"sales_value":"sum"})\
  .sort_values("sales_value", ascending=False)\

print(man_trans.head(5))

top_man_id = man_trans.head(1)["manufacturer_id"].item()

print(top_man_id)

products.loc[products["manufacturer_id"] == top_man_id]\
  .groupby("department")\
  .agg(dep_count = ("department","count"))\
  .sort_values("dep_count", ascending=False)\
  .head(5)


      manufacturer_id  sales_value
54                 69   1262305.37
0                   2    197927.59
657               764     95022.41
88                103     66109.63
1056             1208     63640.63
69


,dep_count
department,
GROCERY,8704
DRUG GM,1657
PASTRY,703
MEAT-PCKGD,421
DELI,418



**Q12:** For each income level, what is the most frequently purchased product category?  

*Hint:* Join demographics → transactions → products, group by income & category, count quantity, get top per income.


In [164]:
# Your code here
income_quant = demographics.merge(transactions, on="household_id", how="inner")\
  .merge(products, on="product_id", how="inner")\
  .groupby(["income","product_category"])\
  .agg({"quantity":"sum"})\
  .reset_index()\
  .sort_values("quantity", ascending=False)
maxidx = income_quant.groupby("income")["quantity"].idxmax()
income_quant.loc[maxidx]


,income,product_category,quantity
69,100-124K,COUPON/MISC ITEMS,6489551
348,125-149K,COUPON/MISC ITEMS,7753726
628,15-24K,COUPON/MISC ITEMS,5157242
908,150-174K,COUPON/MISC ITEMS,6352923
1176,175-199K,COUPON/MISC ITEMS,1843471
1424,200-249K,COUPON/MISC ITEMS,105087
1657,25-34K,COUPON/MISC ITEMS,6570244
1937,250K+,COUPON/MISC ITEMS,1096221
2211,35-49K,COUPON/MISC ITEMS,16342739
2505,50-74K,COUPON/MISC ITEMS,25842845


## Homework Deliverable


- Implement the code to answer the above questions.  
- Once you have all your answers, go to the homework quiz on Canvas and submit your answers.  
- Save your notebook — you'll upload it on Canvas as part of the homework.
